In [1]:
# ============================================================
# RAG CONFIGURATION EVALUATION
# ============================================================
#
# PURPOSE
# -------
# Evaluate the 9 summaries produced by the frozen 3 x 3 RAG
# configuration experiment.
#
# The experimental configurations combine:
#
#   3 chunking strategies:
#       - Whole note
#       - Fixed-size
#       - Section-aware
#
#   x
#
#   3 embedding approaches:
#       - Gemini
#       - BGE
#       - MedCPT
#
# All summaries were generated using:
#
#   Generator: GPT-5.6-Luna
#   Retrieval: Top-20 chunks
#   Evidence ordering: chronological after retrieval
#
#
# EVALUATION STAGES
# -----------------
# Eval 1 — Coverage
#     Measures how much clinically relevant source information
#     is represented in the generated summary.
#
# Eval 2 — TF-IDF similarity
#     Measures lexical similarity between the generated summary
#     and the original clinical record.
#
# Eval 3 — Clinical LLM evaluation
#     Evaluates clinically important properties that lexical
#     metrics alone cannot reliably capture.
#
#
# IMPORTANT
# ---------
# This notebook DOES NOT regenerate RAG summaries.
#
# It loads the frozen summaries produced by:
#     01_configuration_summaries.ipynb
#
# This keeps generation and evaluation separate and prevents
# accidental additional LLM generation calls.
# ============================================================

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
# ...existing code...

In [3]:
# ============================================================
# LOAD FROZEN RAG SUMMARIES
# ============================================================
#
# These are the 9 summaries generated in the configuration
# experiment. They are treated as fixed inputs throughout this
# evaluation notebook.
# ============================================================

SUMMARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "short"
    / "rag_configuration_summaries.json"
)

with SUMMARY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    rag_results = json.load(file)

print("Loaded configurations:", len(rag_results))

for configuration in rag_results:
    print("-", configuration)

Loaded configurations: 9
- Whole + Gemini
- Whole + BGE
- Whole + MedCPT
- Fixed + Gemini
- Fixed + BGE
- Fixed + MedCPT
- Section + Gemini
- Section + BGE
- Section + MedCPT


In [4]:
# ============================================================
# OPENAI EVALUATOR SETUP
# ============================================================

import os

from openai import OpenAI

EVALUATOR_MODEL = "gpt-5.6-luna"

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set.")

openai_client = OpenAI(
    api_key=api_key
)

print("Client class:", type(openai_client).__name__)
print("Client module:", type(openai_client).__module__)
print("Evaluator model:", EVALUATOR_MODEL)

Client class: OpenAI
Client module: openai
Evaluator model: gpt-5.6-luna


In [11]:
# ============================================================
# EVALUATION 1 — REFERENCE-FACT COVERAGE
# ============================================================
#
# PURPOSE
# -------
# Measure how completely each generated RAG summary captures
# the clinically important information contained in the
# complete source clinical record.
#
# Rather than relying on word overlap, coverage is evaluated
# against a frozen checklist of clinically important reference
# facts derived from the complete patient record.
#
#
# SCORING
# -------
# Each reference fact receives:
#
#   2 = Fully captured
#       The clinically important meaning is represented.
#
#   1 = Partially captured
#       The core concept is present, but meaningful clinical
#       information from the reference fact is missing.
#
#   0 = Absent
#       The clinical information is not represented.
#
#
# WHY THIS EVALUATION IS SEPARATE
# -------------------------------
# This metric evaluates INFORMATION COVERAGE only.
#
# It does NOT determine whether information in the summary is
# factually supported by the source record. Unsupported claims
# and other clinical-quality properties are evaluated
# separately in later evaluations.
#
#
# EXPERIMENTAL CONTROL
# --------------------
# - The SAME frozen reference facts are used for all 9
#   RAG configurations.
#
# - The SAME evaluator model and prompt are used for all
#   configurations.
#
# - The 9 generated summaries remain frozen and are not
#   regenerated in this notebook.
# ============================================================

In [47]:
# ============================================================
# FROZEN REFERENCE-FACT CHECKLIST
# ============================================================
#
# IMPORTANT:
# These reference facts are copied unchanged from the original
# RAG configuration experiment.
#
# They must remain fixed across all 9 configurations so that
# every summary is evaluated against the same clinical
# information target.
# ============================================================

In [16]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))
notes = pd.read_csv(DATA_DIR / "raw" / "clinical_notes.csv")
print("Original notes:", len(notes))
print(notes.columns.tolist())
print("Notes:", notes.shape)

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
Notes: (1602, 9)


In [17]:
notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning + deduplication:", len(notes_dedup))

Raw notes: 1602
After cleaning + deduplication: 1103


In [18]:
# ------------------------------------------------------------
# Build the complete short-patient source record used to
# construct the coverage reference checklist.
# ------------------------------------------------------------

SELECTED_PERSON_ID = "c87e610e-ac2a-48cf-ac32-054f3e595498"

reference_notes = (
    notes_dedup[
        notes_dedup["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Reference source notes:", len(reference_notes))

Reference source notes: 10


In [19]:
# Prompt used ONLY to construct the evaluation reference.
# It is deliberately patient-independent so the same procedure
# could later be applied to another patient's source record.

REFERENCE_EXTRACTION_PROMPT = """
You are constructing a reference checklist for evaluating a
longitudinal clinical summary.

Review the complete chronological clinical record provided.

Extract clinically important facts and events necessary to represent
the patient's longitudinal clinical course.

Include:
- major diagnoses and clinically important conditions
- major investigations and their findings
- treatments and important treatment changes
- clinically meaningful changes in symptoms or clinical status
- important referrals, outcomes, and disposition

Rules:
- Include only information explicitly documented in the source record.
- Do not infer or speculate.
- Consolidate repeated information into a single reference fact.
- Preserve temporal relationships when they are explicitly supported.
- Do not treat repeated documentation of the same fact as separate facts.
- Each reference fact must contain only one main clinical claim.
- Return a numbered list of concise, atomic reference facts.
- Do not generate a narrative summary.
"""

In [5]:
# reference_facts = [

#     # ---------- Presentation ----------
#     {
#         "id": "F01",
#         "category": "presentation",
#         "fact": "Patient presented with progressive exertional breathlessness."
#     },
#     {
#         "id": "F02",
#         "category": "presentation",
#         "fact": "Patient was hypoxic at presentation, with SpO2 89% on room air."
#     },

#     # ---------- Diagnosis / investigations ----------
#     {
#         "id": "F03",
#         "category": "diagnosis",
#         "fact": "Chronic thromboembolic pulmonary hypertension (CTEPH) was diagnosed."
#     },
#     {
#         "id": "F04",
#         "category": "investigation",
#         "fact": "CT pulmonary angiography confirmed chronic thromboembolic disease."
#     },
#     {
#         "id": "F05",
#         "category": "investigation",
#         "fact": "Echocardiography demonstrated severe pulmonary hypertension with right ventricular strain."
#     },
#     {
#         "id": "F06",
#         "category": "investigation",
#         "fact": "V/Q scanning demonstrated mismatched perfusion defects consistent with CTEPH."
#     },
#     {
#         "id": "F07",
#         "category": "investigation",
#         "fact": "NT-proBNP was markedly elevated during the clinical course."
#     },

#     # ---------- Treatment ----------
#     {
#         "id": "F08",
#         "category": "treatment",
#         "fact": "Supplemental low-flow oxygen therapy was used to treat hypoxia."
#     },
#     {
#         "id": "F09",
#         "category": "treatment",
#         "fact": "Therapeutic anticoagulation was initiated during the clinical course."
#     },
#     {
#         "id": "F10",
#         "category": "treatment",
#         "fact": "Rivaroxaban was used for ongoing/long-term anticoagulation."
#     },
#     {
#         "id": "F11",
#         "category": "treatment",
#         "fact": "Furosemide was used during management of pulmonary hypertension/right-heart strain and fluid retention."
#     },
#     {
#         "id": "F12",
#         "category": "treatment",
#         "fact": "Sildenafil 20 mg three times daily was introduced for pulmonary hypertension management."
#     },
#     {
#         "id": "F13",
#         "category": "treatment",
#         "fact": "Physiotherapy included breathing exercises and progressive mobilisation."
#     },
#     {
#         "id": "F14",
#         "category": "treatment",
#         "fact": "Energy-conservation/pacing strategies and a home exercise programme were provided."
#     },
#     {
#         "id": "F15",
#         "category": "treatment",
#         "fact": "Mild anaemia was documented later in the course and treated with ferrous sulfate."
#     },

#     # ---------- Clinical progression ----------
#     {
#         "id": "F16",
#         "category": "progression",
#         "fact": "Breathlessness and exercise tolerance improved during the clinical course."
#     },
#     {
#         "id": "F17",
#         "category": "progression",
#         "fact": "Supplemental oxygen requirements decreased, with the patient later maintaining approximately 94-95% SpO2 on room air."
#     },

#     # ---------- Referral / outcome ----------
#     {
#         "id": "F18",
#         "category": "outcome",
#         "fact": "The patient was referred to a tertiary pulmonary hypertension centre for pulmonary endarterectomy assessment."
#     },
#     {
#         "id": "F19",
#         "category": "outcome",
#         "fact": "The patient was ultimately discharged after clinical improvement."
#     },
#     {
#         "id": "F20",
#         "category": "outcome",
#         "fact": "Follow-up at the tertiary pulmonary hypertension centre was planned after discharge."
#     },
# ]

In [ ]:
from src.llm.llm import generate_summary

reference_checklist = generate_summary(
    patient_id=SELECTED_PERSON_ID,
    notes_df=reference_notes,
    prompt=REFERENCE_EXTRACTION_PROMPT
)

print(reference_checklist)



1. On 04/01/26, Ren Suzuki, a 1-year-old female, presented with acute right ear pain.

2. Her parents reported irritability, frequent crying, reduced feeding, sleep difficulty, and right-ear pulling beginning the night before presentation.

3. Parents reported no fever, vomiting, diarrhoea, or other systemic symptoms.

4. Initial observations were within normal limits, including temperature 36.8°C, heart rate 110 bpm, respiratory rate 28 breaths/min, and oxygen saturation 99% on air.

5. Examination showed mild erythema and moderate tenderness of the right outer ear.

6. Otoscopy demonstrated a bulging, mildly erythematous right tympanic membrane with middle-ear effusion.

7. The left tympanic membrane was normal.

8. There were no documented signs of mastoiditis, cellulitis, systemic infection, respiratory distress, neck stiffness, facial asymmetry, or neurological deficit.

9. The diagnosis was acute right otitis media with effusion, described by the specialty registrar as likely bac

In [23]:
# ------------------------------------------------------------
# Freeze the short-patient reference checklist for coverage.
# ------------------------------------------------------------

reference_facts = [
    {"id": "F01", "fact": "On 04/01/26, Ren Suzuki, a 1-year-old female, presented with acute right ear pain."},
    {"id": "F02", "fact": "Her parents reported irritability, frequent crying, reduced feeding, sleep difficulty, and right-ear pulling beginning the night before presentation."},
    {"id": "F03", "fact": "Parents reported no fever, vomiting, diarrhoea, or other systemic symptoms."},
    {"id": "F04", "fact": "Initial observations were within normal limits, including temperature 36.8°C, heart rate 110 bpm, respiratory rate 28 breaths/min, and oxygen saturation 99% on air."},
    {"id": "F05", "fact": "Examination showed mild erythema and moderate tenderness of the right outer ear."},
    {"id": "F06", "fact": "Otoscopy demonstrated a bulging, mildly erythematous right tympanic membrane with middle-ear effusion."},
    {"id": "F07", "fact": "The left tympanic membrane was normal."},
    {"id": "F08", "fact": "There were no documented signs of mastoiditis, cellulitis, systemic infection, respiratory distress, neck stiffness, facial asymmetry, or neurological deficit."},
    {"id": "F09", "fact": "The diagnosis was acute right otitis media with effusion, described by the specialty registrar as likely bacterial in origin."},
    {"id": "F10", "fact": "No laboratory tests, imaging, or other investigations were performed."},
    {"id": "F11", "fact": "Paracetamol 120 mg was administered in the emergency department for pain relief."},
    {"id": "F12", "fact": "The patient was referred to paediatrics/ENT for inpatient admission and accepted under Dr. Lisa Doreen Hayes."},
    {"id": "F13", "fact": "Oral amoxicillin was prescribed for 5 days; the documented regimen included 125 mg/5 mL suspension three times daily, with discharge instructions specifying 2.5 mL three times daily."},
    {"id": "F14", "fact": "Paracetamol 120 mg/5 mL suspension was prescribed as 5 mL four times daily as needed for pain."},
    {"id": "F15", "fact": "Warm compresses and ear hygiene were recommended for symptomatic relief and prevention of irritation."},
    {"id": "F16", "fact": "The pharmacist reviewed and confirmed the amoxicillin prescription as appropriate for the patient’s age and weight, with no medication changes."},
    {"id": "F17", "fact": "During admission, the patient remained afebrile and had no new systemic symptoms."},
    {"id": "F18", "fact": "On subsequent review, the parents reported significant improvement in pain, and the patient tolerated oral antibiotics and analgesia without adverse effects."},
    {"id": "F19", "fact": "Repeat examination showed reduced right tympanic membrane erythema and effusion, with the patient alert, interactive, and comfortable."},
    {"id": "F20", "fact": "The patient was planned for discharge later that day with completion of the 5-day amoxicillin course and as-needed paracetamol."},
    {"id": "F21", "fact": "Parents were advised to seek medical attention for worsening pain, discharge, fever, vomiting, lethargy, or other signs of deterioration."},
    {"id": "F22", "fact": "ENT follow-up was arranged for 2 weeks after discharge."},
    {"id": "F23", "fact": "The discharge treatment plan and ENT follow-up were communicated to the patient’s GP for continuity of care."},
]

print("Reference facts:", len(reference_facts))

Reference facts: 23


In [24]:
print("Reference facts:", len(reference_facts))

for item in reference_facts:
    print(
        f"{item['id']} | {item['fact']}"
    )

Reference facts: 23
F01 | On 04/01/26, Ren Suzuki, a 1-year-old female, presented with acute right ear pain.
F02 | Her parents reported irritability, frequent crying, reduced feeding, sleep difficulty, and right-ear pulling beginning the night before presentation.
F03 | Parents reported no fever, vomiting, diarrhoea, or other systemic symptoms.
F04 | Initial observations were within normal limits, including temperature 36.8°C, heart rate 110 bpm, respiratory rate 28 breaths/min, and oxygen saturation 99% on air.
F05 | Examination showed mild erythema and moderate tenderness of the right outer ear.
F06 | Otoscopy demonstrated a bulging, mildly erythematous right tympanic membrane with middle-ear effusion.
F07 | The left tympanic membrane was normal.
F08 | There were no documented signs of mastoiditis, cellulitis, systemic infection, respiratory distress, neck stiffness, facial asymmetry, or neurological deficit.
F09 | The diagnosis was acute right otitis media with effusion, described b

In [25]:
# ============================================================
# REFERENCE-FACT COVERAGE JUDGE PROMPT
# ============================================================
#
# The evaluator receives:
#   1. The complete frozen reference-fact checklist.
#   2. One generated RAG summary.
#
# It independently assigns a 0/1/2 coverage score to every
# reference fact.
#
# The prompt explicitly separates coverage from faithfulness:
# unsupported information is NOT penalized in this evaluation.
# ============================================================

REFERENCE_COVERAGE_PROMPT = """
You are evaluating the information coverage of a generated clinical summary.

You will receive:
1. A list of reference clinical facts derived from the source clinical record.
2. A generated clinical summary.

For EACH reference fact, determine how completely the clinical information
in that reference fact is represented in the generated summary.

Scoring:
2 = FULLY CAPTURED
    The clinically important meaning of the reference fact is clearly
    represented in the generated summary. Exact wording is not required.

1 = PARTIALLY CAPTURED
    The core clinical concept is represented, but clinically meaningful
    information contained in the reference fact is missing or incomplete.

0 = ABSENT
    The clinical information in the reference fact is not represented
    in the generated summary.

Important rules:
- Evaluate semantic meaning, not exact word overlap.
- Do not reward information merely because it is medically related.
- Do not infer information that the generated summary does not state.
- Score each reference fact independently.
- Do not evaluate whether the generated summary is factually correct here.
  This evaluation measures COVERAGE only.
- Do not penalize additional information in the generated summary.
  Unsupported information will be evaluated separately.
- For every score, provide a short evidence phrase copied or closely
  paraphrased from the generated summary.
- If the score is 0, write "Not represented" as the evidence.

Return ONLY valid JSON in this exact structure:

{
  "scores": [
    {
      "fact_id": "F01",
      "score": 0,
      "evidence": "..."
    }
  ]
}
"""

In [27]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

In [26]:
from openai import OpenAI

from src.llm.llm import LLM_MODEL, LLM_PROVIDER

coverage_client = OpenAI(
    api_key= os.getenv("OPENAI_API_KEY")
)

EVALUATOR_MODEL = LLM_MODEL
EVALUATOR_PROVIDER = LLM_PROVIDER

print("Coverage evaluator provider:", EVALUATOR_PROVIDER)
print("Coverage evaluator model:", EVALUATOR_MODEL)

Coverage evaluator provider: OpenAI
Coverage evaluator model: gpt-5.6-luna


In [28]:
def evaluate_reference_coverage(
    summary: str,
    reference_facts: list[dict],
) -> list[dict]:
    """
    Evaluate one generated clinical summary against the frozen
    reference-fact checklist.

    Each fact receives:
        2 = fully captured
        1 = partially captured
        0 = absent

    Structured output is enforced so the response can be parsed
    reliably without manual JSON cleanup.
    """

    facts_text = "\n".join(
        f"{item['id']}: {item['fact']}"
        for item in reference_facts
    )

    user_message = f"""
REFERENCE FACTS:
{facts_text}

GENERATED SUMMARY:
{summary}
"""

    response = coverage_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": REFERENCE_COVERAGE_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "reference_coverage",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "scores": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "fact_id": {
                                        "type": "string"
                                    },
                                    "score": {
                                        "type": "integer",
                                        "enum": [0, 1, 2],
                                    },
                                    "evidence": {
                                        "type": "string"
                                    },
                                },
                                "required": [
                                    "fact_id",
                                    "score",
                                    "evidence",
                                ],
                                "additionalProperties": False,
                            },
                        }
                    },
                    "required": ["scores"],
                    "additionalProperties": False,
                },
            },
        },
    )

    raw_output = response.choices[0].message.content
    result = json.loads(raw_output)

    return result["scores"]

In [29]:
# ============================================================
# SINGLE-CONFIGURATION SANITY TEST
# ============================================================
#
# Run the evaluator on one configuration before evaluating all
# 9 summaries.
#
# We verify that:
# - valid JSON is returned
# - every frozen reference fact receives a score
# - scores are limited to 0, 1, or 2
# ============================================================

test_configuration = "Whole + Gemini"

test_summary = rag_results[
    test_configuration
]["summary"]

test_coverage = evaluate_reference_coverage(
    summary=test_summary,
    reference_facts=reference_facts,
)

print("Configuration:", test_configuration)
print("Reference facts:", len(reference_facts))
print("Returned scores:", len(test_coverage))

assert len(test_coverage) == len(reference_facts)

assert all(
    item["score"] in {0, 1, 2}
    for item in test_coverage
)

print("Coverage sanity check passed.")

Configuration: Whole + Gemini
Reference facts: 23
Returned scores: 23
Coverage sanity check passed.


In [30]:
# ============================================================
# RUN REFERENCE-FACT COVERAGE — ALL 9 CONFIGURATIONS
# ============================================================
#
# COST / RELIABILITY CONTROLS
# ---------------------------
# - One evaluator call is required per configuration.
# - The already completed Whole + Gemini sanity-test result
#   is reused rather than evaluated again.
# - Each successful result is checkpointed immediately.
# - If this cell is rerun, completed configurations are skipped.
# - Execution stops on an API/parsing error rather than
#   automatically retrying and creating unnecessary calls.
#
# The generated RAG summaries and frozen reference facts are
# NOT modified during this evaluation.
# ============================================================

In [31]:
COVERAGE_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    /"short"
    / "rag_reference_fact_coverage_gpt56luna.json"
)

COVERAGE_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Load previously completed evaluations if they exist.
if COVERAGE_OUTPUT_PATH.exists():
    with COVERAGE_OUTPUT_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        coverage_results = json.load(file)
else:
    coverage_results = {}

In [32]:
# Reuse the successful sanity-test evaluation.
if test_configuration not in coverage_results:
    coverage_results[test_configuration] = test_coverage

    with COVERAGE_OUTPUT_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            coverage_results,
            file,
            indent=2,
            ensure_ascii=False,
        )


In [33]:
import time

for configuration, result in rag_results.items():

    # Avoid paying for an evaluation that already exists.
    if configuration in coverage_results:
        print(f"Skipping completed: {configuration}")
        continue

    print(f"\nEvaluating coverage: {configuration}")

    try:
        scores = evaluate_reference_coverage(
            summary=result["summary"],
            reference_facts=reference_facts,
        )

        # Validate the evaluator output before saving it.
        assert len(scores) == len(reference_facts)

        assert all(
            item["score"] in {0, 1, 2}
            for item in scores
        )

        coverage_results[configuration] = scores

        # Checkpoint immediately after each successful call.
        with COVERAGE_OUTPUT_PATH.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                coverage_results,
                file,
                indent=2,
                ensure_ascii=False,
            )

        print(f"Saved: {configuration}")

    except Exception as error:
        print(
            f"FAILED: {configuration}\n"
            f"{type(error).__name__}: {error}"
        )
        break

    time.sleep(2)


print(
    f"\nCompleted coverage evaluations: "
    f"{len(coverage_results)}/"
    f"{len(rag_results)}"
)

print("Checkpoint:", COVERAGE_OUTPUT_PATH)

Skipping completed: Whole + Gemini

Evaluating coverage: Whole + BGE
Saved: Whole + BGE

Evaluating coverage: Whole + MedCPT
Saved: Whole + MedCPT

Evaluating coverage: Fixed + Gemini
Saved: Fixed + Gemini

Evaluating coverage: Fixed + BGE
Saved: Fixed + BGE

Evaluating coverage: Fixed + MedCPT
Saved: Fixed + MedCPT

Evaluating coverage: Section + Gemini
Saved: Section + Gemini

Evaluating coverage: Section + BGE
Saved: Section + BGE

Evaluating coverage: Section + MedCPT
Saved: Section + MedCPT

Completed coverage evaluations: 9/9
Checkpoint: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/short/rag_reference_fact_coverage_gpt56luna.json


In [15]:
# ============================================================
# COVERAGE RESULTS SUMMARY
# ============================================================
#
# The LLM-based coverage evaluation is now COMPLETE.
#
# This section performs NO LLM calls.
# It only aggregates the previously saved 0/1/2 judgments
# into configuration-level coverage statistics.
#
# For 20 reference facts:
#     Maximum score = 20 x 2 = 40 points
#
# Reported metrics:
# - Coverage Points
# - Coverage %
# - Fully Captured facts
# - Partially Captured facts
# - Absent facts
# ============================================================

coverage_summary = []

max_score = len(reference_facts) * 2

for configuration, scores in coverage_results.items():

    score_values = np.array([
        item["score"]
        for item in scores
    ])

    total_score = score_values.sum()

    coverage_summary.append(
        {
            "Configuration": configuration,
            "Coverage Points": int(total_score),
            "Max Points": max_score,
            "Coverage %": round(
                (total_score / max_score) * 100,
                1,
            ),
            "Fully Captured": int(
                (score_values == 2).sum()
            ),
            "Partially Captured": int(
                (score_values == 1).sum()
            ),
            "Absent": int(
                (score_values == 0).sum()
            ),
        }
    )


coverage_summary_df = pd.DataFrame(
    coverage_summary
)

coverage_summary_df = (
    coverage_summary_df
    .sort_values(
        "Coverage %",
        ascending=False,
    )
    .reset_index(drop=True)
)

coverage_summary_df

,Configuration,Coverage Points,Max Points,Coverage %,Fully Captured,Partially Captured,Absent
0,Whole + Gemini,0,40,0.0,0,0,20
1,Whole + BGE,0,40,0.0,0,0,20
2,Whole + MedCPT,0,40,0.0,0,0,20
3,Fixed + Gemini,0,40,0.0,0,0,20
4,Fixed + BGE,0,40,0.0,0,0,20
5,Fixed + MedCPT,0,40,0.0,0,0,20
6,Section + Gemini,0,40,0.0,0,0,20
7,Section + BGE,0,40,0.0,0,0,20
8,Section + MedCPT,0,40,0.0,0,0,20


In [36]:
# ============================================================
# COVERAGE RESULT — INITIAL OBSERVATION
# ============================================================
#
# Whole-note configurations achieved the strongest reference-
# fact coverage overall. Whole + Gemini captured all 20 frozen
# reference facts (100%), while Whole + BGE and Fixed + BGE
# each achieved 97.5%.
#
# Section-aware configurations showed lower coverage,
# particularly Section + Gemini (52.5%). Because retrieval was
# fixed at Top-20 chunks, finer-grained section chunking also
# resulted in substantially smaller retrieved contexts.
#
# These results measure information coverage ONLY and should
# not be interpreted as the final configuration ranking.
# Faithfulness, temporal consistency, and other evaluation
# dimensions are assessed separately.
# ============================================================

In [50]:
# ============================================================
# EVALUATION 2 — TF-IDF COSINE SIMILARITY
# ============================================================
#
# PURPOSE
# -------
# Measure lexical similarity between each generated RAG summary
# and the complete source clinical record.
#
# For each configuration:
#
#   1. Fit TF-IDF jointly on:
#        - the complete source clinical record
#        - the generated summary
#
#   2. Represent both texts in the same TF-IDF vector space.
#
#   3. Calculate cosine similarity between the source-record
#      vector and the generated-summary vector.
#
#
# INTERPRETATION
# --------------
# Higher similarity indicates greater lexical/content overlap
# with the complete source record.
#
# This metric does NOT directly measure:
# - clinical factuality
# - information importance
# - temporal correctness
# - semantic equivalence when different terminology is used
#
# It therefore complements, rather than replaces, the
# reference-fact coverage evaluation.
#
#
# IMPORTANT
# ---------
# This evaluation is deterministic and makes NO LLM/API calls.
# ============================================================

In [54]:
NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes = pd.read_csv(NOTES_PATH)

In [74]:
PATIENT_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

In [77]:
# ------------------------------------------------------------
# Clean and deduplicate the source notes using the same
# preprocessing applied in the RAG configuration experiment.
# ------------------------------------------------------------

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first",
    )
    .reset_index(drop=True)
)

patient_notes = (
    notes_dedup[
        notes_dedup["person_id"] == PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Patient:", PATIENT_ID)
print("Patient notes:", len(patient_notes))

Patient: c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Patient notes: 45


In [78]:
# ------------------------------------------------------------
# Build the complete source record for TF-IDF evaluation.
#
# Use the same 45 cleaned and deduplicated patient notes used
# in the RAG configuration experiment.
# ------------------------------------------------------------

source_text = "\n\n".join(
    patient_notes["clean_note_text"]
    .astype(str)
    .tolist()
)

print("Patient:", PATIENT_ID)
print("Source notes:", len(patient_notes))
print("Source characters:", len(source_text))
print("Source words:", len(source_text.split()))

Patient: c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Source notes: 45
Source characters: 35863
Source words: 5134


In [58]:
# ============================================================
# BUILD COMPLETE SOURCE RECORD FOR TF-IDF EVALUATION
# ============================================================
#
# TF-IDF compares each generated RAG summary against the
# COMPLETE clinical record for the evaluation patient.
#
# IMPORTANT:
# - This is NOT a retrieved Top-20 RAG context.
# - It contains all clinical notes available for this patient.
# - Notes are ordered chronologically before concatenation.
# - This same source record is used for all 9 configurations.
#
# No LLM/API calls are made here.
# ============================================================

In [79]:
tfidf_results = []

for configuration, result in rag_results.items():

    summary = result["summary"]

    # Fit TF-IDF jointly on the complete source record and
    # this configuration's generated summary.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
    )

    vectors = vectorizer.fit_transform(
        [
            source_text,
            summary,
        ]
    )

    # Compare the complete source-record vector with the
    # generated-summary vector.
    similarity = cosine_similarity(
        vectors[0:1],
        vectors[1:2],
    )[0][0]

    tfidf_results.append(
        {
            "Configuration": configuration,
            "TF-IDF Cosine Similarity": similarity,
        }
    )


tfidf_similarity_df = pd.DataFrame(
    tfidf_results
)

tfidf_similarity_df = (
    tfidf_similarity_df
    .sort_values(
        "TF-IDF Cosine Similarity",
        ascending=False,
    )
    .reset_index(drop=True)
)

tfidf_similarity_df

,Configuration,TF-IDF Cosine Similarity
0,Whole + MedCPT,0.396356
1,Section + BGE,0.386688
2,Whole + BGE,0.377191
3,Fixed + MedCPT,0.373188
4,Fixed + Gemini,0.370844
5,Whole + Gemini,0.367396
6,Fixed + BGE,0.367226
7,Section + MedCPT,0.347418
8,Section + Gemini,0.257903


In [80]:
print("Source characters:", len(source_text))
print("Source words:", len(source_text.split()))

Source characters: 35863
Source words: 5134


In [81]:
# ============================================================
# SAVE TF-IDF EVALUATION RESULTS
# ============================================================
#
# Evaluation 2 is complete.
#
# These scores were calculated deterministically and required
# no LLM/API calls.
#
# The output is saved separately so downstream ranking and
# analysis can reuse it without recomputing the evaluation.
# ============================================================

TFIDF_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_tfidf_similarity_gpt56luna_top20.csv"
)

tfidf_similarity_df.to_csv(
    TFIDF_OUTPUT_PATH,
    index=False,
)

print("Saved:", TFIDF_OUTPUT_PATH)

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag_tfidf_similarity_gpt56luna_top20.csv


In [64]:
# ============================================================
# TEMPORAL EVALUATION MODEL
# ============================================================
#
# GPT-5.6 Luna is used as the LLM judge for semantic event
# alignment in the regenerated evaluation pipeline.
#
# The summaries being evaluated were also generated with
# GPT-5.6 Luna; however, generation and evaluation are separate
# API calls with separate prompts and purposes.
#
# The consolidated reference timeline is reused from the
# original experiment because it was derived exclusively from
# the unchanged source clinical record, not from generated
# summaries.
# ============================================================

In [82]:
# ============================================================
# EVALUATION 4 — TEMPORAL CONSISTENCY
#
# Measures whether matched clinical events appear in the
# generated summaries in the same relative order as the source.
# ============================================================

EVALUATOR_MODEL = "gpt-5.6-luna"

rag_summaries = {
    configuration: result["summary"]
    for configuration, result in rag_results.items()
}

print("Evaluator model:", EVALUATOR_MODEL)
print("RAG summaries:", len(rag_summaries))
print("Patient notes:", len(patient_notes))

Evaluator model: gpt-5.6-luna
RAG summaries: 9
Patient notes: 45


In [83]:
# ------------------------------------------------------------
# OpenAI client used for all LLM-based temporal evaluation.
# ------------------------------------------------------------

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set.")

openai_client = OpenAI(
    api_key=api_key
)

print("Client:", type(openai_client).__name__)
print("Evaluator:", EVALUATOR_MODEL)

Client: OpenAI
Evaluator: gpt-5.6-luna


In [84]:
# ------------------------------------------------------------
# Give the 45 source notes an authoritative chronological index.
# ------------------------------------------------------------

temporal_notes = (
    patient_notes
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
    .copy()
)

temporal_notes["source_note_index"] = temporal_notes.index

print("Temporal notes:", len(temporal_notes))
print(
    "Index range:",
    temporal_notes["source_note_index"].min(),
    "to",
    temporal_notes["source_note_index"].max(),
)

Temporal notes: 45
Index range: 0 to 44


In [85]:
# ------------------------------------------------------------
# Candidate-event extraction prompt.
#
# Keep this prompt identical to the original temporal
# evaluation methodology.
# ------------------------------------------------------------

TEMPORAL_EVENT_PROMPT = """
Extract the clinically meaningful CURRENT events from this clinical note
for longitudinal temporal evaluation.

The source notes have already been ordered using creation_timestamp.
Do NOT use dates written inside the note to establish chronology.

Extract events that occur, are newly observed, are newly decided, or
represent a change in clinical state at THIS point in the record.

Include:
- new clinical findings or changes in patient status
- investigations ordered, performed, or newly reviewed
- new diagnoses or diagnostic conclusions
- treatments started, stopped, changed, or continued when clinically meaningful
- referrals or disposition decisions
- meaningful rehabilitation or management progression

Do NOT extract:
- background medical history or allergies
- staff/administrative information
- repeated historical information merely restated from earlier care
- earlier events mentioned only as context
- dates or times as separate events
- duplicate statements of the same event within the note

Distinguish plans from completed actions.
For example, "admission planned" must remain a plan and must not become
"patient admitted."

Use only information explicitly stated in the note.
Keep each event concise and patient-specific.

Return valid JSON only:

{
  "events": [
    "event 1",
    "event 2"
  ]
}
"""

In [86]:
def extract_temporal_events(note_text):
    """
    Extract candidate clinical events from one source note.
    """

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": TEMPORAL_EVENT_PROMPT,
            },
            {
                "role": "user",
                "content": note_text,
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "temporal_events",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "events": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            },
                        }
                    },
                    "required": ["events"],
                    "additionalProperties": False,
                },
            },
        },
    )

    result = json.loads(
        response.choices[0].message.content
    )

    return result["events"]

In [87]:
# ------------------------------------------------------------
# Extract candidate temporal events from all 45 source notes.
#
# Each event retains its source_note_index and timestamp.
# ------------------------------------------------------------

temporal_events = []
failed_notes = []

for i, row in temporal_notes.iterrows():

    try:
        events = extract_temporal_events(
            row["clean_note_text"]
        )

    except Exception as error:
        failed_notes.append(
            int(row["source_note_index"])
        )

        print(
            f"{i + 1:02d}/45 | "
            f"NOTE {row['source_note_index']} | "
            f"FAILED | {type(error).__name__}"
        )

        break

    for event in events:
        temporal_events.append(
            {
                "source_note_index": int(
                    row["source_note_index"]
                ),
                "creation_timestamp": row[
                    "creation_timestamp"
                ],
                "event": event,
            }
        )

    print(
        f"{i + 1:02d}/45 | "
        f"NOTE {row['source_note_index']} | "
        f"{len(events)} events"
    )

print("\nTotal candidate events:", len(temporal_events))
print("Failed notes:", failed_notes)

01/45 | NOTE 0 | 4 events
02/45 | NOTE 1 | 4 events
03/45 | NOTE 2 | 4 events
04/45 | NOTE 3 | 4 events
05/45 | NOTE 4 | 9 events
06/45 | NOTE 5 | 10 events
07/45 | NOTE 6 | 6 events
08/45 | NOTE 7 | 4 events
09/45 | NOTE 8 | 2 events
10/45 | NOTE 9 | 5 events
11/45 | NOTE 10 | 6 events
12/45 | NOTE 11 | 3 events
13/45 | NOTE 12 | 7 events
14/45 | NOTE 13 | 4 events
15/45 | NOTE 14 | 2 events
16/45 | NOTE 15 | 4 events
17/45 | NOTE 16 | 7 events
18/45 | NOTE 17 | 6 events
19/45 | NOTE 18 | 6 events
20/45 | NOTE 19 | 4 events
21/45 | NOTE 20 | 3 events
22/45 | NOTE 21 | 8 events
23/45 | NOTE 22 | 6 events
24/45 | NOTE 23 | 3 events
25/45 | NOTE 24 | 5 events
26/45 | NOTE 25 | 4 events
27/45 | NOTE 26 | 6 events
28/45 | NOTE 27 | 6 events
29/45 | NOTE 28 | 4 events
30/45 | NOTE 29 | 2 events
31/45 | NOTE 30 | 3 events
32/45 | NOTE 31 | 6 events
33/45 | NOTE 32 | 6 events
34/45 | NOTE 33 | 4 events
35/45 | NOTE 34 | 2 events
36/45 | NOTE 35 | 3 events
37/45 | NOTE 36 | 8 events
38/45 | NO

In [89]:
# ------------------------------------------------------------
# Save candidate events so extraction never needs to be rerun.
# ------------------------------------------------------------

temporal_events_df = pd.DataFrame(
    temporal_events
)

TEMPORAL_EVENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "temporal_candidate_events_gpt56luna.csv"
)

temporal_events_df.to_csv(
    TEMPORAL_EVENTS_PATH,
    index=False,
)

print("Candidate events:", len(temporal_events_df))
print("Saved:", TEMPORAL_EVENTS_PATH)

Candidate events: 217
Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/temporal_candidate_events_gpt56luna.csv


In [90]:
# ------------------------------------------------------------
# Consolidate repeated events while preserving genuine
# clinical progression.
# ------------------------------------------------------------

TEMPORAL_CONSOLIDATION_PROMPT = """
You are consolidating clinical events for longitudinal temporal evaluation.

The source clinical notes have already been ordered by creation_timestamp.
The event's source note index therefore represents its authoritative
chronological position.

For each CURRENT candidate event, determine whether it represents:

KEEP:
- a genuinely new clinical event
- a new investigation, treatment, diagnosis, referral, or disposition
- a meaningful change in clinical state
- a new value/finding that represents clinical progression
- a treatment being started, stopped, changed, or meaningfully continued

REMOVE:
- the same clinical event already represented in EARLIER events
- historical information merely restated
- duplicated findings/results with no new clinical change
- administrative/non-clinically meaningful information

Important:
Do NOT remove genuine progression simply because it concerns the same
clinical concept.

Example:
"SpO2 89% on room air"
"SpO2 improved to 92% after oxygen"
"SpO2 later 94% on room air"
are separate meaningful states and should all be kept.

But repeated statements of the same NT-proBNP result of 4500 pg/mL,
without a new measurement or change, should not create multiple events.

Do not use dates written inside event text to establish chronology.
Use the supplied source note indices only.

Return valid JSON only:

{
  "decisions": [
    {
      "candidate_id": 0,
      "decision": "KEEP" or "REMOVE",
      "reason": "brief reason"
    }
  ]
}
"""

In [91]:
def consolidate_note_events(
    current_events,
    earlier_events,
):
    """
    Decide which current events represent new temporal events
    versus repeated information.
    """

    current_candidates = [
        {
            "candidate_id": i,
            "event": event,
        }
        for i, event in enumerate(current_events)
    ]

    user_message = f"""
EARLIER CONSOLIDATED EVENTS:
{json.dumps(earlier_events, indent=2)}

CURRENT CANDIDATE EVENTS:
{json.dumps(current_candidates, indent=2)}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": TEMPORAL_CONSOLIDATION_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ],
        response_format={"type": "json_object"},
    )

    raw_output = response.choices[0].message.content

    if raw_output is None or not raw_output.strip():
        raise ValueError(
            "Evaluator returned an empty response."
        )

    result = json.loads(raw_output)

    return result["decisions"]

In [92]:
# ------------------------------------------------------------
# Build the consolidated reference timeline note-by-note.
# ------------------------------------------------------------

consolidated_events = []
consolidation_decisions = []

for note_index in sorted(
    temporal_events_df["source_note_index"].unique()
):

    current_df = temporal_events_df[
        temporal_events_df["source_note_index"]
        == note_index
    ]

    current_events = current_df["event"].tolist()

    decisions = consolidate_note_events(
        current_events=current_events,
        earlier_events=[
            item["event"]
            for item in consolidated_events
        ],
    )

    kept_count = 0

    for decision in decisions:

        candidate_id = decision["candidate_id"]
        event_text = current_events[candidate_id]

        consolidation_decisions.append(
            {
                "source_note_index": int(note_index),
                "event": event_text,
                "decision": decision["decision"],
                "reason": decision["reason"],
            }
        )

        if decision["decision"] == "KEEP":
            consolidated_events.append(
                {
                    "source_note_index": int(note_index),
                    "event": event_text,
                }
            )

            kept_count += 1

    print(
        f"NOTE {note_index:02d} | "
        f"{len(current_events)} candidates | "
        f"{kept_count} kept"
    )

NOTE 00 | 4 candidates | 4 kept
NOTE 01 | 4 candidates | 2 kept
NOTE 02 | 4 candidates | 1 kept
NOTE 03 | 4 candidates | 2 kept
NOTE 04 | 9 candidates | 7 kept
NOTE 05 | 10 candidates | 6 kept
NOTE 06 | 6 candidates | 1 kept
NOTE 07 | 4 candidates | 3 kept
NOTE 08 | 2 candidates | 0 kept
NOTE 09 | 5 candidates | 4 kept
NOTE 10 | 6 candidates | 3 kept
NOTE 11 | 3 candidates | 1 kept
NOTE 12 | 7 candidates | 4 kept
NOTE 13 | 4 candidates | 2 kept
NOTE 14 | 2 candidates | 2 kept
NOTE 15 | 4 candidates | 0 kept
NOTE 16 | 7 candidates | 1 kept
NOTE 17 | 6 candidates | 2 kept
NOTE 18 | 6 candidates | 2 kept
NOTE 19 | 4 candidates | 1 kept
NOTE 20 | 3 candidates | 1 kept
NOTE 21 | 8 candidates | 2 kept
NOTE 22 | 6 candidates | 1 kept
NOTE 23 | 3 candidates | 1 kept
NOTE 24 | 5 candidates | 4 kept
NOTE 25 | 4 candidates | 1 kept
NOTE 26 | 6 candidates | 1 kept
NOTE 27 | 6 candidates | 2 kept
NOTE 28 | 4 candidates | 1 kept
NOTE 29 | 2 candidates | 1 kept
NOTE 30 | 3 candidates | 2 kept
NOTE 31

In [93]:
# ------------------------------------------------------------
# Save the consolidated reference timeline and decisions.
# ------------------------------------------------------------

CONSOLIDATED_EVENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "temporal_consolidated_events_gpt56luna.csv"
)

CONSOLIDATION_DECISIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "temporal_consolidation_decisions_gpt56luna.csv"
)

pd.DataFrame(
    consolidated_events
).to_csv(
    CONSOLIDATED_EVENTS_PATH,
    index=False,
)

pd.DataFrame(
    consolidation_decisions
).to_csv(
    CONSOLIDATION_DECISIONS_PATH,
    index=False,
)

print(
    "Final consolidated events:",
    len(consolidated_events),
)

Final consolidated events: 83


In [94]:
reference_timeline = [
    {
        "reference_event_id": int(i),
        "source_note_index": int(
            event["source_note_index"]
        ),
        "event": event["event"],
    }
    for i, event in enumerate(
        consolidated_events
    )
]

print(
    "Reference events:",
    len(reference_timeline),
)

Reference events: 83


In [96]:
# ------------------------------------------------------------
# Split generated summaries into ordered sentence-level units.
# ------------------------------------------------------------

import re

def split_summary_into_units(summary):
    """
    Split one summary into ordered text units.
    """

    text = re.sub(
        r"\s+",
        " ",
        summary.strip(),
    )

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text,
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    return [
        {
            "summary_unit": i,
            "text": sentence,
        }
        for i, sentence in enumerate(
            sentences,
            start=1,
        )
    ]


summary_units = {
    config: split_summary_into_units(summary)
    for config, summary in rag_summaries.items()
}

for config, units in summary_units.items():
    print(
        f"{config}: {len(units)} units"
    )

Whole + Gemini: 28 units
Whole + BGE: 36 units
Whole + MedCPT: 21 units
Fixed + Gemini: 19 units
Fixed + BGE: 33 units
Fixed + MedCPT: 39 units
Section + Gemini: 20 units
Section + BGE: 26 units
Section + MedCPT: 13 units


In [97]:
# ------------------------------------------------------------
# Map each reference event to the summary unit representing it.
# ------------------------------------------------------------

SUMMARY_EVENT_MAPPING_PROMPT = """
You are aligning a generated longitudinal clinical summary with a
reference timeline of clinical events.

The reference events are already in authoritative chronological order,
based on the source clinical notes' creation_timestamp.

For each reference event, determine whether the underlying clinical event
is represented in the generated summary.

MATCH:
- The summary clearly expresses the same underlying clinical event.
- Paraphrasing is allowed.
- Exact wording is not required.
- A summary sentence may represent more than one reference event.

NO_MATCH:
- The event is absent from the summary.
- The summary only contains a vague related statement that does not
  establish the reference event.
- The clinical content is materially different.

Important:
- Do NOT judge whether the summary event occurs in the correct chronological
  position. Your task is only to identify correspondence and where the
  matched event appears in the summary.
- Do NOT penalize omissions.
- Do NOT judge hallucinations or unsupported claims.
- Do NOT use outside medical knowledge.
- Do NOT use dates inside the reference events to change their reference
  chronology.
- Match only on the clinical content of the event.

The generated summary is divided into numbered units in the order they
appear. For every matched reference event, return the number of the
summary unit that represents it.

Return valid JSON only:

{
  "matches": [
    {
      "reference_event_id": 0,
      "match": "MATCH",
      "summary_unit": 3
    },
    {
      "reference_event_id": 1,
      "match": "NO_MATCH",
      "summary_unit": null
    }
  ]
}
"""

In [98]:
def map_reference_events_to_summary(
    reference_timeline,
    units,
):
    """
    Map reference clinical events to generated summary units.
    """

    summary_for_model = [
        {
            "summary_unit": unit["summary_unit"],
            "text": unit["text"],
        }
        for unit in units
    ]

    user_message = f"""
REFERENCE TIMELINE:
{json.dumps(reference_timeline, indent=2)}

GENERATED SUMMARY UNITS:
{json.dumps(summary_for_model, indent=2)}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": SUMMARY_EVENT_MAPPING_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ],
        response_format={"type": "json_object"},
    )

    result = json.loads(
        response.choices[0].message.content
    )

    return result["matches"]

In [99]:
test_config = "Whole + Gemini"

test_matches = map_reference_events_to_summary(
    reference_timeline,
    summary_units[test_config],
)

print(
    "Returned decisions:",
    len(test_matches),
)

matched = [
    item
    for item in test_matches
    if item["match"] == "MATCH"
]

print(
    "Matched reference events:",
    len(matched),
)

Returned decisions: 83
Matched reference events: 57


In [100]:
assert len(test_matches) == len(reference_timeline)

print("Temporal mapping sanity check passed.")

Temporal mapping sanity check passed.


In [102]:
# ------------------------------------------------------------
# Map the reference timeline to all 9 summaries.
# Completed configurations are checkpointed and skipped later.
# ------------------------------------------------------------

TEMPORAL_MAPPING_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_temporal_mappings_gpt56luna.json"
)

if TEMPORAL_MAPPING_PATH.exists():
    with TEMPORAL_MAPPING_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        temporal_mapping_results = json.load(file)
else:
    temporal_mapping_results = {}


# Reuse the successful sanity-test call.
if test_config not in temporal_mapping_results:
    temporal_mapping_results[
        test_config
    ] = test_matches


for config, units in summary_units.items():

    if config in temporal_mapping_results:
        print(
            f"Skipping completed: {config}"
        )
        continue

    print(f"Mapping: {config}")

    matches = map_reference_events_to_summary(
        reference_timeline,
        units,
    )

    if len(matches) != len(reference_timeline):
        raise ValueError(
            f"{config}: expected "
            f"{len(reference_timeline)} decisions, "
            f"got {len(matches)}"
        )

    temporal_mapping_results[
        config
    ] = matches

    with TEMPORAL_MAPPING_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            temporal_mapping_results,
            file,
            indent=2,
        )

    matched_count = sum(
        item["match"] == "MATCH"
        for item in matches
    )

    print(
        f"Decisions: {len(matches)} | "
        f"Matched: {matched_count}"
    )

Skipping completed: Whole + Gemini
Mapping: Whole + BGE
Decisions: 83 | Matched: 65
Mapping: Whole + MedCPT
Decisions: 83 | Matched: 64
Mapping: Fixed + Gemini
Decisions: 83 | Matched: 58
Mapping: Fixed + BGE
Decisions: 83 | Matched: 54
Mapping: Fixed + MedCPT
Decisions: 83 | Matched: 44
Mapping: Section + Gemini
Decisions: 83 | Matched: 28
Mapping: Section + BGE
Decisions: 83 | Matched: 47
Mapping: Section + MedCPT
Decisions: 83 | Matched: 26


In [104]:
# ------------------------------------------------------------
# Calculate pairwise temporal-order accuracy.
#
# No LLM calls occur here.
# ------------------------------------------------------------
from itertools import combinations

def calculate_temporal_consistency(matches):
    """
    Calculate pairwise temporal-order accuracy for matched events.
    """

    matched_events = [
        {
            "reference_event_id": int(
                item["reference_event_id"]
            ),
            "summary_unit": int(
                item["summary_unit"]
            ),
        }
        for item in matches
        if item["match"] == "MATCH"
        and item["summary_unit"] is not None
    ]

    matched_events = sorted(
        matched_events,
        key=lambda item: item[
            "reference_event_id"
        ],
    )

    correct_pairs = 0
    reversed_pairs = 0
    tied_pairs = 0

    for event_a, event_b in combinations(
        matched_events,
        2,
    ):

        unit_a = event_a["summary_unit"]
        unit_b = event_b["summary_unit"]

        if unit_a < unit_b:
            correct_pairs += 1

        elif unit_a > unit_b:
            reversed_pairs += 1

        else:
            tied_pairs += 1

    comparable_pairs = (
        correct_pairs
        + reversed_pairs
    )

    if comparable_pairs > 0:
        temporal_score = (
            correct_pairs
            / comparable_pairs
        ) * 100
    else:
        temporal_score = None

    return {
        "matched_events": len(
            matched_events
        ),
        "correct_pairs": correct_pairs,
        "reversed_pairs": reversed_pairs,
        "tied_pairs_excluded": tied_pairs,
        "comparable_pairs": comparable_pairs,
        "temporal_consistency": temporal_score,
    }

In [105]:
temporal_results = []

for config, matches in (
    temporal_mapping_results.items()
):

    result = calculate_temporal_consistency(
        matches
    )

    temporal_results.append(
        {
            "Configuration": config,
            **result,
        }
    )

temporal_results_df = pd.DataFrame(
    temporal_results
)

temporal_results_df = (
    temporal_results_df
    .sort_values(
        "temporal_consistency",
        ascending=False,
    )
    .reset_index(drop=True)
)

temporal_results_df

,Configuration,matched_events,correct_pairs,reversed_pairs,tied_pairs_excluded,comparable_pairs,temporal_consistency
0,Fixed + MedCPT,44,850,77,19,927,91.693635
1,Fixed + BGE,54,1259,134,38,1393,90.380474
2,Whole + Gemini,57,1373,169,54,1542,89.040208
3,Section + BGE,47,928,116,37,1044,88.888889
4,Whole + BGE,65,1723,284,73,2007,85.849527
5,Fixed + Gemini,58,1294,262,97,1556,83.161954
6,Whole + MedCPT,64,1526,373,117,1899,80.358083
7,Section + MedCPT,26,224,59,42,283,79.151943
8,Section + Gemini,28,255,76,47,331,77.039275


In [106]:
TEMPORAL_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "rag_temporal_consistency_gpt56luna.csv"
)

temporal_results_df.to_csv(
    TEMPORAL_RESULTS_PATH,
    index=False,
)

print("Saved:", TEMPORAL_RESULTS_PATH)

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/rag_temporal_consistency_gpt56luna.csv


In [107]:
# ============================================================
# EVALUATION 3 — CLAIM-LEVEL FAITHFULNESS
#
# Measures whether clinical claims in each generated summary
# are supported by evidence in the 45-note source record.
#
# Pipeline:
# summary -> atomic claims -> source evidence -> support judgment
# ============================================================

EVALUATOR_MODEL = "gpt-5.6-luna"

print("Evaluator model:", EVALUATOR_MODEL)
print("RAG summaries:", len(rag_summaries))
print("Source notes:", len(patient_notes))

Evaluator model: gpt-5.6-luna
RAG summaries: 9
Source notes: 45


In [113]:
# ------------------------------------------------------------
# Atomic-claim extraction prompt.
#
# Keep this prompt identical to the original Eval 3 method so
# the new summaries are decomposed in the same way as before.
# ------------------------------------------------------------

CLAIM_EXTRACTION_PROMPT = """
You are extracting atomic clinical claims from a generated longitudinal
clinical summary.

An atomic clinical claim is ONE independently verifiable patient-specific
clinical assertion.

Rules:
- Extract only information explicitly stated in the generated summary.
- Do not add, infer, correct, or reinterpret information.
- Each claim must contain only ONE independently verifiable assertion.
- Split statements containing multiple clinical assertions into separate claims.
- Preserve clinically important information such as diagnoses, symptoms,
  investigation findings, treatments, medications, doses, clinical
  progression, and outcomes.
- Write each claim concisely while preserving its clinical meaning.
- Extract each distinct clinical assertion ONLY ONCE, even if the same
  information is repeated elsewhere in the summary.
- If a later statement merely restates an earlier claim (for example in a
  "Key diagnoses" or concluding section), do not extract it again.
- Do not extract headings or formatting.
- Do not evaluate whether a claim is correct.
- Do not compare claims with the source record.
- Do not extract dates or temporal information as separate claims.
  Temporal consistency will be evaluated separately.

Return ONLY valid JSON in this exact structure:

{
  "claims": [
    "...",
    "...",
    "..."
  ]
}
"""

In [116]:
# ------------------------------------------------------------
# Extract atomic clinical claims from one generated summary.
#
# Claim IDs are assigned locally so the evaluator only needs
# to return the extracted claim text.
# ------------------------------------------------------------

def extract_atomic_claims(summary):

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": CLAIM_EXTRACTION_PROMPT,
            },
            {
                "role": "user",
                "content": summary,
            },
        ],
        max_completion_tokens=4000,
    )

    raw_output = response.choices[0].message.content

    try:
        result = json.loads(raw_output)

        claims = [
            {
                "claim_id": f"C{i:02d}",
                "claim": claim,
            }
            for i, claim in enumerate(
                result["claims"],
                start=1,
            )
        ]

        return claims

    except json.JSONDecodeError as error:
        print(
            "Finish reason:",
            response.choices[0].finish_reason,
        )
        print("JSON error:", error)
        print("Last 300 characters:")
        print(repr(raw_output[-300:]))

        return None

In [117]:
# ------------------------------------------------------------
# Sanity-test atomic claim extraction on one summary.
# ------------------------------------------------------------

test_config = "Whole + Gemini"

test_claims = extract_atomic_claims(
    rag_summaries[test_config]
)

print("Configuration:", test_config)
print("Atomic claims:", len(test_claims))

for item in test_claims[:10]:
    print(
        item["claim_id"],
        "|",
        item["claim"],
    )

Configuration: Whole + Gemini
Atomic claims: 63
C01 | Abena Bonsu is a 43-year-old woman with hypertension.
C02 | She has asthma.
C03 | She has a nut allergy.
C04 | She reported no regular medications.
C05 | She presented with two weeks of progressive exertional breathlessness.
C06 | She was tachycardic.
C07 | She was tachypnoeic.
C08 | She was borderline hypotensive.
C09 | Her oxygen saturation was 89–90% on room air.
C10 | She required low-flow oxygen.


In [118]:
# ------------------------------------------------------------
# Extract atomic claims for all 9 RAG summaries.
#
# Results are saved after each configuration so completed
# extractions do not need to be repeated.
# ------------------------------------------------------------

ATOMIC_CLAIMS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "rag_atomic_claims_gpt56luna.json"
)

# Start from an existing checkpoint if one exists.
if ATOMIC_CLAIMS_PATH.exists():

    with ATOMIC_CLAIMS_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        claim_results = json.load(file)

else:
    claim_results = {}


# Reuse the successful Whole + Gemini sanity test.
claim_results[test_config] = test_claims

with ATOMIC_CLAIMS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        claim_results,
        file,
        indent=2,
    )


# Extract claims only for configurations not already completed.
for config_name, summary in rag_summaries.items():

    if config_name in claim_results:
        print(
            f"Skipping completed: {config_name} | "
            f"{len(claim_results[config_name])} claims"
        )
        continue

    print(f"\nExtracting claims: {config_name}")

    claims = extract_atomic_claims(summary)

    if claims is None:
        print(f"FAILED: {config_name}")
        break

    claim_results[config_name] = claims

    # Save immediately after every successful configuration.
    with ATOMIC_CLAIMS_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            claim_results,
            file,
            indent=2,
        )

    print(
        f"Saved: {config_name} | "
        f"{len(claims)} claims"
    )


print(
    "\nCompleted configurations:",
    f"{len(claim_results)}/9",
)

print("Checkpoint:", ATOMIC_CLAIMS_PATH)

Skipping completed: Whole + Gemini | 63 claims

Extracting claims: Whole + BGE
Saved: Whole + BGE | 87 claims

Extracting claims: Whole + MedCPT
Saved: Whole + MedCPT | 77 claims

Extracting claims: Fixed + Gemini
Saved: Fixed + Gemini | 55 claims

Extracting claims: Fixed + BGE
Saved: Fixed + BGE | 47 claims

Extracting claims: Fixed + MedCPT
Saved: Fixed + MedCPT | 43 claims

Extracting claims: Section + Gemini
Saved: Section + Gemini | 48 claims

Extracting claims: Section + BGE
Saved: Section + BGE | 58 claims

Extracting claims: Section + MedCPT
Saved: Section + MedCPT | 42 claims

Completed configurations: 9/9
Checkpoint: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/rag_atomic_claims_gpt56luna.json


In [119]:
# ------------------------------------------------------------
# Check the number of atomic claims extracted from each summary.
# ------------------------------------------------------------

total_claims = 0

for config_name, claims in claim_results.items():

    print(
        f"{config_name}: "
        f"{len(claims)} claims"
    )

    total_claims += len(claims)

print("\nTotal atomic claims:", total_claims)

Whole + Gemini: 63 claims
Whole + BGE: 87 claims
Whole + MedCPT: 77 claims
Fixed + Gemini: 55 claims
Fixed + BGE: 47 claims
Fixed + MedCPT: 43 claims
Section + Gemini: 48 claims
Section + BGE: 58 claims
Section + MedCPT: 42 claims

Total atomic claims: 520


In [120]:
# ------------------------------------------------------------
# Prepare the 45 source notes used for faithfulness evidence.
# ------------------------------------------------------------

reference_notes = (
    patient_notes
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

source_evidence_texts = (
    reference_notes["clean_note_text"]
    .astype(str)
    .tolist()
)

print("Source evidence notes:", len(source_evidence_texts))

Source evidence notes: 45


In [121]:
# ------------------------------------------------------------
# Prepare the 45 source notes used as the evidence corpus
# for claim-level faithfulness evaluation.
# ------------------------------------------------------------

reference_notes = (
    patient_notes
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
    .copy()
)

source_evidence_texts = (
    reference_notes["clean_note_text"]
    .astype(str)
    .tolist()
)

print("Source evidence notes:", len(source_evidence_texts))

Source evidence notes: 45


In [122]:
# ------------------------------------------------------------
# Gemini client and embedding model used for evidence retrieval.
# ------------------------------------------------------------
from google import genai

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set.")

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"

print("Gemini client initialized.")
print("Embedding model:", GEMINI_EMBEDDING_MODEL)

Gemini client initialized.
Embedding model: gemini-embedding-001


In [123]:
# ------------------------------------------------------------
# Gemini embedding helper used for faithfulness evidence
# retrieval. Texts are embedded individually to respect the
# available Gemini API endpoint and rate limits.
# ------------------------------------------------------------

def embed_gemini_texts(texts):
    """
    Generate Gemini embeddings one text at a time.
    """
    embeddings = []

    for index, text in enumerate(texts, start=1):

        print(
            f"Gemini embedding: "
            f"{index}/{len(texts)}"
        )

        response = gemini_client.models.embed_content(
            model=GEMINI_EMBEDDING_MODEL,
            contents=text,
        )

        embedding = (
            response.embeddings[0].values
        )

        embeddings.append(embedding)

        # Avoid sending embedding requests too quickly.
        time.sleep(1.0)

    return np.array(embeddings)

In [124]:
# ------------------------------------------------------------
# Embed the 45 source notes with Gemini.
#
# These embeddings form the evidence index used to retrieve
# the most relevant source notes for each atomic claim.
# ------------------------------------------------------------

print("Generating Gemini embeddings for source evidence...")

source_gemini_embeddings = embed_gemini_texts(
    source_evidence_texts
)

print(
    "Source embedding shape:",
    source_gemini_embeddings.shape
)

Generating Gemini embeddings for source evidence...
Gemini embedding: 1/45
Gemini embedding: 2/45
Gemini embedding: 3/45
Gemini embedding: 4/45
Gemini embedding: 5/45
Gemini embedding: 6/45
Gemini embedding: 7/45
Gemini embedding: 8/45
Gemini embedding: 9/45
Gemini embedding: 10/45
Gemini embedding: 11/45
Gemini embedding: 12/45
Gemini embedding: 13/45
Gemini embedding: 14/45
Gemini embedding: 15/45
Gemini embedding: 16/45
Gemini embedding: 17/45
Gemini embedding: 18/45
Gemini embedding: 19/45
Gemini embedding: 20/45
Gemini embedding: 21/45
Gemini embedding: 22/45
Gemini embedding: 23/45
Gemini embedding: 24/45
Gemini embedding: 25/45
Gemini embedding: 26/45
Gemini embedding: 27/45
Gemini embedding: 28/45
Gemini embedding: 29/45
Gemini embedding: 30/45
Gemini embedding: 31/45
Gemini embedding: 32/45
Gemini embedding: 33/45
Gemini embedding: 34/45
Gemini embedding: 35/45
Gemini embedding: 36/45
Gemini embedding: 37/45
Gemini embedding: 38/45
Gemini embedding: 39/45
Gemini embedding: 40/

In [125]:
# ------------------------------------------------------------
# Retrieve the five source notes most semantically similar
# to one atomic claim using Gemini embeddings.
#
# The 45 source-note embeddings are reused for every claim.
# ------------------------------------------------------------

from sklearn.metrics.pairwise import cosine_similarity


def retrieve_evidence_gemini(claim, top_k=5):

    # Embed only the current claim.
    claim_embedding = embed_gemini_texts(
        [claim]
    )

    # Compare the claim with all 45 source notes.
    similarities = cosine_similarity(
        claim_embedding,
        source_gemini_embeddings
    )[0]

    # Rank source notes from most to least similar.
    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    retrieved = []

    for rank, note_index in enumerate(
        top_indices,
        start=1
    ):

        retrieved.append({
            "rank": rank,
            "note_index": int(note_index),
            "similarity": float(
                similarities[note_index]
            ),
            "evidence": source_evidence_texts[
                note_index
            ],
        })

    return retrieved

In [126]:
# ------------------------------------------------------------
# Sanity-check Gemini Top-5 evidence retrieval on one
# extracted claim before processing the full claim set.
# ------------------------------------------------------------

test_claim = claim_results[
    "Whole + Gemini"
][0]["claim"]

test_evidence = retrieve_evidence_gemini(
    test_claim,
    top_k=5
)

print("CLAIM:")
print(test_claim)

print("\nTOP-5 RETRIEVED SOURCE NOTES:")

for item in test_evidence:

    print(
        f"Rank {item['rank']} | "
        f"Note {item['note_index']} | "
        f"Similarity {item['similarity']:.4f}"
    )

Gemini embedding: 1/1
CLAIM:
Abena Bonsu is a 43-year-old woman with hypertension.

TOP-5 RETRIEVED SOURCE NOTES:
Rank 1 | Note 28 | Similarity 0.7348
Rank 2 | Note 0 | Similarity 0.7254
Rank 3 | Note 13 | Similarity 0.7227
Rank 4 | Note 4 | Similarity 0.7125
Rank 5 | Note 30 | Similarity 0.6865


In [127]:
# ------------------------------------------------------------
# Faithfulness verifier.
#
# The evaluator receives one atomic claim and its Top-5
# Gemini-retrieved source notes and determines whether the
# source evidence supports the clinical content of the claim.
# ------------------------------------------------------------

FAITHFULNESS_PROMPT = """
Evaluate whether one clinical-summary claim is supported by the provided
source evidence.

Judge only source-grounded CLINICAL CONTENT.

Temporal consistency is evaluated separately. Ignore temporal placement
such as "on arrival", "later", or "on discharge day" when assigning the
faithfulness label. Judge whether the underlying clinical fact is supported.

SUPPORTED:
The evidence explicitly states or sufficiently establishes the clinical claim.

UNSUPPORTED:
The clinical claim, value, diagnosis, treatment, investigation, finding,
or other substantive clinical content is not established by the evidence.

For compound claims, all substantive clinical components must be supported.

Use only the provided evidence. Do not use outside knowledge or infer
undocumented information.

Return valid JSON only:
{
  "label": "SUPPORTED" or "UNSUPPORTED",
  "reason": "brief source-based explanation",
  "evidence_quote": "shortest relevant source text, or null if unsupported"
}
"""

In [128]:
def verify_claim_with_llm(claim, retrieved_evidence):
    """
    Verify one atomic claim against its Top-5 retrieved
    source notes using the frozen faithfulness evaluator.
    """

    evidence_text = "\n\n".join(
        [
            f"[SOURCE NOTE {item['note_index']}]\n"
            f"{item['evidence']}"
            for item in retrieved_evidence
        ]
    )

    user_message = f"""
CLAIM:
{claim}

SOURCE EVIDENCE:
{evidence_text}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": FAITHFULNESS_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ],
        response_format={"type": "json_object"},
    )

    raw_output = response.choices[0].message.content

    if raw_output is None or not raw_output.strip():
        raise ValueError(
            "Evaluator returned an empty response."
        )

    return json.loads(raw_output)

In [129]:
test_verification = verify_claim_with_llm(
    test_claim,
    test_evidence
)

print("CLAIM:")
print(test_claim)

print("\nVERIFICATION:")
print(
    json.dumps(
        test_verification,
        indent=2
    )
)

CLAIM:
Abena Bonsu is a 43-year-old woman with hypertension.

VERIFICATION:
{
  "label": "SUPPORTED",
  "reason": "Source note 0 explicitly identifies Abena Bonsu as a 43-year-old female and states that her past medical history includes hypertension (HTN).",
  "evidence_quote": "Patient Abena Bonsu, a 43-year-old female... Past medical history includes HTN"
}


In [130]:
# ------------------------------------------------------------
# Faithfulness results are checkpointed after every claim so
# completed Gemini retrieval + LLM verification is not repeated.
# ------------------------------------------------------------

FAITHFULNESS_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "rag_claim_faithfulness_gpt56luna.json"
)

if FAITHFULNESS_RESULTS_PATH.exists():

    with FAITHFULNESS_RESULTS_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        faithfulness_results = json.load(file)

else:
    faithfulness_results = {}

print(
    "Existing configurations:",
    len(faithfulness_results)
)

Existing configurations: 0


In [131]:
# ------------------------------------------------------------
# Save the successful Whole + Gemini C01 sanity-test result
# before starting the full faithfulness evaluation.
# ------------------------------------------------------------

test_claim_item = claim_results[
    "Whole + Gemini"
][0]

faithfulness_results.setdefault(
    "Whole + Gemini",
    {}
)

faithfulness_results[
    "Whole + Gemini"
][test_claim_item["claim_id"]] = {
    "claim": test_claim_item["claim"],
    "label": test_verification["label"],
    "reason": test_verification["reason"],
    "evidence_quote": test_verification["evidence_quote"],
    "retrieved_note_indices": [
        item["note_index"]
        for item in test_evidence
    ],
}

with FAITHFULNESS_RESULTS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        faithfulness_results,
        file,
        indent=2,
    )

print(
    "Saved sanity test:",
    "Whole + Gemini",
    test_claim_item["claim_id"],
)

Saved sanity test: Whole + Gemini C01


In [136]:
# ------------------------------------------------------------
# Evaluate every atomic claim against its Top-5 Gemini-retrieved
# source notes.
#
# Each completed claim is saved immediately. If the run stops,
# rerunning this cell skips claims already in the checkpoint.
# ------------------------------------------------------------

for config_name, claims in claim_results.items():

    faithfulness_results.setdefault(
        config_name,
        {}
    )

    print(
        f"\n{'=' * 70}\n"
        f"{config_name} | {len(claims)} claims\n"
        f"{'=' * 70}"
    )

    for claim_item in claims:

        claim_id = claim_item["claim_id"]
        claim_text = claim_item["claim"]

        # Never repeat a successfully completed claim.
        if claim_id in faithfulness_results[config_name]:

            print(
                f"Skipping completed: "
                f"{config_name} | {claim_id}"
            )

            continue

        print(
            f"Evaluating: "
            f"{config_name} | {claim_id}"
        )

        try:
            # Retrieve the five most relevant source notes.
            evidence = retrieve_evidence_gemini(
                claim_text,
                top_k=5,
            )

            # Verify this individual claim against its evidence.
            verification = verify_claim_with_llm(
                claim_text,
                evidence,
            )

            label = verification.get("label")

            if label not in {
                "SUPPORTED",
                "UNSUPPORTED",
            }:
                raise ValueError(
                    f"Unexpected label: {label}"
                )

            # Save the result for this individual claim.
            faithfulness_results[
                config_name
            ][claim_id] = {
                "claim": claim_text,
                "label": label,
                "reason": verification.get(
                    "reason"
                ),
                "evidence_quote": verification.get(
                    "evidence_quote"
                ),
                "retrieved_note_indices": [
                    item["note_index"]
                    for item in evidence
                ],
            }

            # Checkpoint immediately.
            with FAITHFULNESS_RESULTS_PATH.open(
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    faithfulness_results,
                    file,
                    indent=2,
                )

            print(
                f"Saved: {config_name} | "
                f"{claim_id} | {label}"
            )

            # Keep Gemini requests comfortably separated.
            time.sleep(1.0)

        except Exception as error:

            print(
                f"\nSTOPPED at "
                f"{config_name} | {claim_id}"
            )

            print(
                type(error).__name__,
                ":",
                error,
            )

            # Stop rather than silently skipping a claim.
            raise


Whole + Gemini | 63 claims
Skipping completed: Whole + Gemini | C01
Skipping completed: Whole + Gemini | C02
Skipping completed: Whole + Gemini | C03
Skipping completed: Whole + Gemini | C04
Skipping completed: Whole + Gemini | C05
Skipping completed: Whole + Gemini | C06
Skipping completed: Whole + Gemini | C07
Skipping completed: Whole + Gemini | C08
Skipping completed: Whole + Gemini | C09
Skipping completed: Whole + Gemini | C10
Skipping completed: Whole + Gemini | C11
Skipping completed: Whole + Gemini | C12
Skipping completed: Whole + Gemini | C13
Skipping completed: Whole + Gemini | C14
Skipping completed: Whole + Gemini | C15
Skipping completed: Whole + Gemini | C16
Skipping completed: Whole + Gemini | C17
Skipping completed: Whole + Gemini | C18
Skipping completed: Whole + Gemini | C19
Skipping completed: Whole + Gemini | C20
Skipping completed: Whole + Gemini | C21
Skipping completed: Whole + Gemini | C22
Skipping completed: Whole + Gemini | C23
Skipping completed: Whole + G

In [133]:
# ------------------------------------------------------------
# Save the source-note Gemini embeddings so they can be reused
# without calling the Gemini API again.
# ------------------------------------------------------------

SOURCE_EMBEDDINGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "faithfulness_source_gemini_embeddings.npy"
)

np.save(
    SOURCE_EMBEDDINGS_PATH,
    source_gemini_embeddings
)

print("Saved:", SOURCE_EMBEDDINGS_PATH)
print("Shape:", source_gemini_embeddings.shape)

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/faithfulness_source_gemini_embeddings.npy
Shape: (45, 3072)


In [134]:
from dotenv import load_dotenv

# Reload values from .env and overwrite existing environment variables.
load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set.")

# Recreate only the Gemini client with the reloaded key.
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini credentials reloaded.")

Gemini credentials reloaded.


In [135]:
# ------------------------------------------------------------
# Confirm that the reloaded Gemini credentials can generate
# an embedding before resuming the full evaluation.
# ------------------------------------------------------------

key_test_embedding = embed_gemini_texts(
    ["clinical evidence retrieval test"]
)

print("Test embedding shape:", key_test_embedding.shape)

Gemini embedding: 1/1
Test embedding shape: (1, 3072)


In [137]:
# ------------------------------------------------------------
# Final save of all claim-level faithfulness judgments.
# The evaluation loop already checkpointed each completed
# claim; this writes the complete in-memory result once more.
# ------------------------------------------------------------

with FAITHFULNESS_RESULTS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        faithfulness_results,
        file,
        indent=2,
    )

print("Final faithfulness results saved.")
print("Path:", FAITHFULNESS_RESULTS_PATH)

Final faithfulness results saved.
Path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/rag_claim_faithfulness_gpt56luna.json


In [138]:
# ------------------------------------------------------------
# Verify that every extracted atomic claim has a saved
# faithfulness judgment before calculating final scores.
# ------------------------------------------------------------

expected_total = 0
completed_total = 0

for config_name, claims in claim_results.items():

    expected = len(claims)

    completed = len(
        faithfulness_results.get(
            config_name,
            {}
        )
    )

    expected_total += expected
    completed_total += completed

    print(
        f"{config_name}: "
        f"{completed}/{expected}"
    )

print(
    "\nTotal completed:",
    f"{completed_total}/{expected_total}"
)

Whole + Gemini: 63/63
Whole + BGE: 87/87
Whole + MedCPT: 77/77
Fixed + Gemini: 55/55
Fixed + BGE: 47/47
Fixed + MedCPT: 43/43
Section + Gemini: 48/48
Section + BGE: 58/58
Section + MedCPT: 42/42

Total completed: 520/520


In [139]:
# ------------------------------------------------------------
# Summarize claim-level faithfulness for each RAG configuration.
#
# Faithfulness % =
# supported atomic claims / total atomic claims * 100
# ------------------------------------------------------------

faithfulness_summary = []

for config_name, claims in claim_results.items():

    config_results = faithfulness_results[
        config_name
    ]

    total_claims = len(claims)

    supported = sum(
        result["label"] == "SUPPORTED"
        for result in config_results.values()
    )

    unsupported = sum(
        result["label"] == "UNSUPPORTED"
        for result in config_results.values()
    )

    faithfulness_percentage = (
        supported / total_claims
    ) * 100

    faithfulness_summary.append({
        "Configuration": config_name,
        "Total Claims": total_claims,
        "Supported": supported,
        "Unsupported": unsupported,
        "Faithfulness %": round(
            faithfulness_percentage,
            2
        ),
    })


faithfulness_summary_df = pd.DataFrame(
    faithfulness_summary
)

faithfulness_summary_df = (
    faithfulness_summary_df
    .sort_values(
        "Faithfulness %",
        ascending=False,
    )
    .reset_index(drop=True)
)

faithfulness_summary_df

,Configuration,Total Claims,Supported,Unsupported,Faithfulness %
0,Section + BGE,58,52,6,89.66
1,Fixed + Gemini,55,48,7,87.27
2,Fixed + MedCPT,43,36,7,83.72
3,Whole + MedCPT,77,62,15,80.52
4,Fixed + BGE,47,37,10,78.72
5,Section + MedCPT,42,33,9,78.57
6,Whole + Gemini,63,49,14,77.78
7,Whole + BGE,87,66,21,75.86
8,Section + Gemini,48,33,15,68.75


In [140]:
# ------------------------------------------------------------
# Save final configuration-level faithfulness results.
# ------------------------------------------------------------

FAITHFULNESS_SUMMARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "rag_faithfulness_summary_gpt56luna.csv"
)

faithfulness_summary_df.to_csv(
    FAITHFULNESS_SUMMARY_PATH,
    index=False
)

print("Saved:", FAITHFULNESS_SUMMARY_PATH)

faithfulness_summary_df

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/rag_faithfulness_summary_gpt56luna.csv


,Configuration,Total Claims,Supported,Unsupported,Faithfulness %
0,Section + BGE,58,52,6,89.66
1,Fixed + Gemini,55,48,7,87.27
2,Fixed + MedCPT,43,36,7,83.72
3,Whole + MedCPT,77,62,15,80.52
4,Fixed + BGE,47,37,10,78.72
5,Section + MedCPT,42,33,9,78.57
6,Whole + Gemini,63,49,14,77.78
7,Whole + BGE,87,66,21,75.86
8,Section + Gemini,48,33,15,68.75


In [141]:
# ------------------------------------------------------------
# Show all DataFrames currently available in the notebook.
# ------------------------------------------------------------

[
    name
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
]

['_',
 '__',
 '___',
 'coverage_summary_df',
 '_35',
 'notes',
 'notes_clean',
 'notes_dedup',
 'patient_notes',
 'tfidf_similarity_df',
 '_59',
 'temporal_notes',
 '_68',
 '_79',
 'temporal_events_df',
 'current_df',
 'temporal_results_df',
 '_105',
 'reference_notes',
 'faithfulness_summary_df',
 '_139',
 '_140']

In [142]:
# ============================================================
# MASTER RAG EVALUATION TABLE
#
# Combines the final results for:
#   1. Reference-fact coverage
#   2. Claim-level faithfulness
#   3. Temporal consistency
#
# TF-IDF similarity is intentionally excluded from the primary
# evaluation table because it is retained only as a descriptive
# lexical-similarity analysis.
# ============================================================

master_evaluation_df = (
    coverage_summary_df[
        [
            "Configuration",
            "Coverage %",
            "Fully Captured",
            "Partially Captured",
            "Absent",
        ]
    ]
    .merge(
        faithfulness_summary_df[
            [
                "Configuration",
                "Total Claims",
                "Supported",
                "Unsupported",
                "Faithfulness %",
            ]
        ],
        on="Configuration",
        how="inner",
    )
    .merge(
        temporal_results_df[
            [
                "Configuration",
                "matched_events",
                "temporal_consistency",
            ]
        ],
        on="Configuration",
        how="inner",
    )
)

# Rename temporal columns for readability.
master_evaluation_df = master_evaluation_df.rename(
    columns={
        "matched_events": "Matched Temporal Events",
        "temporal_consistency": "Temporal Consistency %",
    }
)

# Sort by configuration so all nine are easy to compare.
configuration_order = [
    "Whole + Gemini",
    "Whole + BGE",
    "Whole + MedCPT",
    "Fixed + Gemini",
    "Fixed + BGE",
    "Fixed + MedCPT",
    "Section + Gemini",
    "Section + BGE",
    "Section + MedCPT",
]

master_evaluation_df["Configuration"] = pd.Categorical(
    master_evaluation_df["Configuration"],
    categories=configuration_order,
    ordered=True,
)

master_evaluation_df = (
    master_evaluation_df
    .sort_values("Configuration")
    .reset_index(drop=True)
)

master_evaluation_df

,Configuration,Coverage %,Fully Captured,Partially Captured,Absent,Total Claims,Supported,Unsupported,Faithfulness %,Matched Temporal Events,Temporal Consistency %
0,Whole + Gemini,100.0,20,0,0,63,49,14,77.78,57,89.040208
1,Whole + BGE,97.5,19,1,0,87,66,21,75.86,65,85.849527
2,Whole + MedCPT,95.0,18,2,0,77,62,15,80.52,64,80.358083
3,Fixed + Gemini,92.5,17,3,0,55,48,7,87.27,58,83.161954
4,Fixed + BGE,97.5,19,1,0,47,37,10,78.72,54,90.380474
5,Fixed + MedCPT,90.0,17,2,1,43,36,7,83.72,44,91.693635
6,Section + Gemini,52.5,9,3,8,48,33,15,68.75,28,77.039275
7,Section + BGE,87.5,16,3,1,58,52,6,89.66,47,88.888889
8,Section + MedCPT,72.5,12,5,3,42,33,9,78.57,26,79.151943


In [143]:
# ------------------------------------------------------------
# Add supplementary TF-IDF similarity to the master table.
# ------------------------------------------------------------

final_evaluation_df = master_evaluation_df.merge(
    tfidf_similarity_df[
        [
            "Configuration",
            "TF-IDF Cosine Similarity"
        ]
    ],
    on="Configuration",
    how="left"
)

final_evaluation_df

,Configuration,Coverage %,Fully Captured,Partially Captured,Absent,Total Claims,Supported,Unsupported,Faithfulness %,Matched Temporal Events,Temporal Consistency %,TF-IDF Cosine Similarity
0,Whole + Gemini,100.0,20,0,0,63,49,14,77.78,57,89.040208,0.367396
1,Whole + BGE,97.5,19,1,0,87,66,21,75.86,65,85.849527,0.377191
2,Whole + MedCPT,95.0,18,2,0,77,62,15,80.52,64,80.358083,0.396356
3,Fixed + Gemini,92.5,17,3,0,55,48,7,87.27,58,83.161954,0.370844
4,Fixed + BGE,97.5,19,1,0,47,37,10,78.72,54,90.380474,0.367226
5,Fixed + MedCPT,90.0,17,2,1,43,36,7,83.72,44,91.693635,0.373188
6,Section + Gemini,52.5,9,3,8,48,33,15,68.75,28,77.039275,0.257903
7,Section + BGE,87.5,16,3,1,58,52,6,89.66,47,88.888889,0.386688
8,Section + MedCPT,72.5,12,5,3,42,33,9,78.57,26,79.151943,0.347418


In [144]:
# ============================================================
# PRIORITY-BASED CONFIGURATION RANKING
#
# Evaluation priority:
#   1. Faithfulness
#   2. Temporal consistency
#   3. Coverage
#   4. TF-IDF similarity (supplementary)
#
# No averaging or composite score is used.
# ============================================================

priority_ranked_df = (
    final_evaluation_df
    .sort_values(
        by=[
            "Faithfulness %",
            "Temporal Consistency %",
            "Coverage %",
            "TF-IDF Cosine Similarity",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

priority_ranked_df.insert(
    0,
    "Priority Rank",
    range(1, len(priority_ranked_df) + 1)
)

priority_ranked_df[
    [
        "Priority Rank",
        "Configuration",
        "Faithfulness %",
        "Temporal Consistency %",
        "Coverage %",
        "TF-IDF Cosine Similarity",
    ]
]

,Priority Rank,Configuration,Faithfulness %,Temporal Consistency %,Coverage %,TF-IDF Cosine Similarity
0,1,Section + BGE,89.66,88.888889,87.5,0.386688
1,2,Fixed + Gemini,87.27,83.161954,92.5,0.370844
2,3,Fixed + MedCPT,83.72,91.693635,90.0,0.373188
3,4,Whole + MedCPT,80.52,80.358083,95.0,0.396356
4,5,Fixed + BGE,78.72,90.380474,97.5,0.367226
5,6,Section + MedCPT,78.57,79.151943,72.5,0.347418
6,7,Whole + Gemini,77.78,89.040208,100.0,0.367396
7,8,Whole + BGE,75.86,85.849527,97.5,0.377191
8,9,Section + Gemini,68.75,77.039275,52.5,0.257903


In [145]:
PRIORITY_RANKING_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "rag_final_priority_ranking.csv"
)

priority_ranked_df.to_csv(
    PRIORITY_RANKING_PATH,
    index=False
)

print("Saved:", PRIORITY_RANKING_PATH)

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/rag_final_priority_ranking.csv
